# 10분 만에 Registry 구축 — 관리자 설정 및 IAM 거버넌스 가이드

> **⚠️ 주의:** 이 리포지토리에서 제공하는 예제는 실험 및 교육 목적으로만 사용됩니다. 개념과 기법을 보여 주기 위한 것이며 프로덕션 환경에서 직접 사용하도록 설계되지 않았습니다.

## 개요

조직이 수십 개 팀에서 운영하는 MCP 도구 서버, A2A 에이전트, 사용자 지정 스킬 등 AI 에이전트를 대규모로 도입할 때는 검증되지 않은 에이전트가 프로덕션에 진입하지 못하도록 하고 승인된 에이전트를 검색할 수 있게 해 주는 중앙 집중식 거버넌스 레지스트리가 필요합니다.

AWS Agent Registry는 모든 레코드가 검색 가능해지기 전에 승인 워크플로(`DRAFT → PENDING_APPROVAL → APPROVED / REJECTED`)를 통과해야 하는 거버넌스 우선 접근 방식을 제공합니다. 이 튜토리얼에서는 IT/DevOps 관리자가 처음부터 레지스트리를 구축하고, 세 가지 페르소나(관리자, 게시자, 소비자)의 IAM 정책을 구성하고, 세 가지 레코드 유형을 모두 등록하고, 거버넌스 가드레일의 작동을 입증하고, 시맨틱 검색을 검증하는 과정을 10분 이내에 살펴봅니다.

![관리자 설정 및 IAM 거버넌스 아키텍처](images/quick-setup-architecture.png)

### 작동 방식

1. **관리자가 `autoApproval: false`인 Registry를 생성합니다.** 검토 없이는 아무것도 활성화되지 않습니다.
2. **IAM 정책**은 각 페르소나가 허용된 작업만 수행하도록 범위를 지정합니다. 게시자는 자신이 제출한 항목을 승인할 수 없고 소비자는 읽기 전용입니다.
3. **게시자가 레코드(MCP, A2A, CUSTOM)를 등록합니다.** 모든 레코드는 `DRAFT` 상태로 시작합니다.
4. **게시자가 승인을 요청합니다.** 레코드는 `PENDING_APPROVAL`로 전환됩니다.
5. **거버넌스 테스트로 경계를 입증합니다.** 게시자의 자체 승인은 거부되고 소비자의 쓰기 액세스도 거부됩니다.
6. **관리자가 레코드를 승인합니다.** 레코드는 `APPROVED` 상태가 되고 시맨틱 검색으로 찾을 수 있게 됩니다.
7. **소비자가 검색합니다.** 자연어 쿼리는 승인되고 검증된 에이전트를 반환합니다.

### 페르소나

| 페르소나       | 가능한 작업                                                        | 불가능한 작업                                |
|:--------------|:--------------------------------------------------------------|:-----------------------------------------|
| 관리자         | 레지스트리 생성/삭제, 레코드 승인/거부/사용 중단 처리            | 없음                                      |
| 게시자         | 레코드 생성, 승인 요청, DRAFT 레코드 업데이트                    | 레코드 승인/거부, 레지스트리 생성/삭제       |
| 소비자         | 레코드 나열, 가져오기, 검색                                      | 항목 생성, 수정 또는 승인                    |

### 지원되는 레코드 유형

| 유형     | 설명                                      | 설명자              |
|:---------|:-------------------------------------------------|:-------------------------|
| MCP      | Model Context Protocol 서버(도구)           | `server` + `tools`       |
| A2A      | Agent-to-Agent 프로토콜 에이전트             | `agentCard`              |
| CUSTOM   | 스킬, 사용자 지정 API 리소스 및 기타 항목      | `custom`                 |

## 튜토리얼 세부 정보

| 정보              | 세부 정보                                                                |
|:-------------------------|:-----------------------------------------------------------------------|
| 튜토리얼 유형            | 대화형                                                            |
| AgentCore components     | AWS Agent Registry                                                     |
| 레코드 유형              | MCP, A2A, CUSTOM                                                       |
| 승인 모드                | 수동(`autoApproval: false`)                                         |
| 튜토리얼 구성 요소       | AWS Agent Registry, AWS IAM                                            |
| 튜토리얼 분야            | 여러 분야에 적용 가능(모든 엔터프라이즈 에이전트 거버넌스 워크플로에 적용 가능) |
| 예제 난이도              | 초급                                                               |
| 사용 SDK                 | boto3                                                                  |

## 튜토리얼 주요 기능

* 수동 승인 워크플로(`DRAFT → PENDING_APPROVAL → APPROVED / REJECTED`)를 사용하는 거버넌스 우선 Agent Registry
* 세 가지 페르소나(관리자, 게시자, 소비자)를 위한 IAM 정책 구성
* 직무 분리를 입증하는 페르소나별 가드레일 테스트(게시자는 자체 승인 불가, 소비자는 쓰기 불가)
* 세 가지 레코드 유형을 한 번에 등록: MCP 서버, A2A 에이전트(인라인 agent card), CUSTOM 스킬
* 승인 후 data plane을 통한 시맨틱 검색 검증
* 프로덕션 준비 체크리스트 및 문제 해결 FAQ
* 생성한 모든 리소스(레코드, 레지스트리, IAM 사용자) 완전 정리

## 사전 요구 사항

- 적절한 권한이 있는 IAM 자격 증명(`IAM_PERMISSIONS.md` 참조). 이 튜토리얼에서는 레지스트리, IAM 사용자, 인라인 정책을 생성하기 위한 관리자 수준의 권한이 필요합니다.

  | 서비스 | 권한 |
  |:--------|:------------|
  | **AWS Agent Registry** | `CreateRegistry`, `DeleteRegistry`, `GetRegistry`, `ListRegistries`, `CreateRegistryRecord`, `DeleteRegistryRecord`, `GetRegistryRecord`, `ListRegistryRecords`, `UpdateRegistryRecord`, `SubmitRegistryRecordForApproval`, `UpdateRegistryRecordStatus`, `SearchRegistryRecords` |
  | **AWS IAM** | `CreateUser`, `DeleteUser`, `PutUserPolicy`, `DeleteUserPolicy`, `CreateAccessKey`, `DeleteAccessKey`, `ListAccessKeys` |
  | **AWS STS** | `GetCallerIdentity` |

- `boto3 >= 1.42.87`이 설치된 Python 3.8+
- 기본 리전(`us-west-2`)이 구성된 AWS CLI

## 생성되는 AWS 리소스

| 리소스                        | 유형                              | 용도                                          |
|:--------------------------------|:----------------------------------|:-------------------------------------------------|
| Agent Registry                  | `bedrock-agentcore:registry`      | 수동 승인 워크플로를 사용하는 중앙 레지스트리   |
| MCP Server 레코드               | `bedrock-agentcore:record`        | 코드 검토 MCP 도구 서버(DRAFT → APPROVED)   |
| A2A Agent 레코드                | `bedrock-agentcore:record`        | 인라인 카드가 있는 규정 준수 에이전트(DRAFT → APPROVED) |
| CUSTOM Skill 레코드             | `bedrock-agentcore:record`        | 데이터 파이프라인 스킬(DRAFT → APPROVED)            |
| IAM 사용자(관리자)              | `AWS::IAM::User`                  | 전체 레지스트리 액세스 + 승인 권한         |
| IAM 사용자(게시자)              | `AWS::IAM::User`                  | 레코드 생성/제출, 승인 권한 없음      |
| IAM 사용자(소비자)              | `AWS::IAM::User`                  | 읽기 전용 액세스 + 시맨틱 검색                |

---
## 클라이언트 설정

In [ ]:
%pip install "boto3>=1.42.87" --quiet

In [ ]:
import os
import boto3
import json
import time
from botocore.exceptions import ClientError

# 구성 - 환경에 맞게 업데이트
AWS_REGION = "us-west-2"

# Amazon SageMaker Notebook을 사용하지 않는 경우 AWS 자격 증명 설정
os.environ["AWS_PROFILE"] = "default"  # SageMaker에서는 이 줄을 주석 처리

# boto3 Session 생성
session = boto3.Session(region_name=AWS_REGION)
ACCOUNT_ID = session.client("sts").get_caller_identity()["Account"]

# Control plane 클라이언트(관리자 작업)
cp_client = session.client("bedrock-agentcore-control")
# Data plane 클라이언트(검색 작업)
dp_client = session.client("bedrock-agentcore")
iam_client = session.client("iam")


def pp(response):
    """API 응답에서 ResponseMetadata를 제외하고 보기 좋게 출력합니다."""
    data = {k: v for k, v in response.items() if k != "ResponseMetadata"}
    print(json.dumps(data, indent=2, default=str))


print(f"Session ready | Region: {AWS_REGION} | Account: {ACCOUNT_ID}")

### 헬퍼

In [ ]:
# 터미널 출력용 ANSI 색상
class C:
    GREEN = "\033[92m"
    RED = "\033[91m"
    YELLOW = "\033[93m"
    CYAN = "\033[96m"
    BOLD = "\033[1m"
    DIM = "\033[2m"
    RESET = "\033[0m"


def wait_for_registry_ready(client, registry_id, timeout=120):
    """레지스트리가 READY 상태가 될 때까지 폴링합니다."""
    start = time.time()
    while time.time() - start < timeout:
        r = client.get_registry(registryId=registry_id)
        if r["status"] == "READY":
            elapsed = int(time.time() - start)
            print(f"  {C.GREEN}✅ Registry READY{C.RESET} {C.DIM}({elapsed}s){C.RESET}")
            return r
        print(f"  {C.YELLOW}⏳ Status: {r['status']}...{C.RESET}")
        time.sleep(3)
    raise TimeoutError(f"Registry not READY after {timeout}s")


def test_action(desc, fn):
    """fn()을 실행하고 성공하면 ALLOWED, AccessDenied이면 DENIED를 출력합니다."""
    try:
        result = fn()
        print(f"  {C.GREEN}✅ ALLOWED:{C.RESET} {desc}")
        return result
    except ClientError as e:
        code = e.response["Error"]["Code"]
        print(f"  {C.RED}🚫 DENIED:{C.RESET}  {desc} {C.DIM}({code}){C.RESET}")
        return None


# 정리 및 표시를 위해 모든 레코드 ID와 이름 추적
RECORD_IDS = {}
RECORD_NAMES = {}


def mask_account(acct):
    """표시용 계정 ID를 마스킹합니다: 1234****5678"""
    if len(acct) >= 8:
        return acct[:4] + "****" + acct[-4:]
    return "****"


print(
    f"{C.GREEN}✅ Session ready{C.RESET} | Region: {C.CYAN}{AWS_REGION}{C.RESET} | Account: {C.CYAN}{mask_account(ACCOUNT_ID)}{C.RESET}"
)

---
## 1단계 — Registry 생성(거버넌스 우선)

관리자가 가장 먼저 수행할 작업은 `autoApproval: false`인 **Registry**를 생성하는 것입니다.
이 설정을 통해 모든 레코드가 검색 가능해지기 전에 반드시 승인 워크플로를 거치게 됩니다.

> **중요**: `CreateRegistry` 호출 후 레지스트리는 `CREATING` 상태가 됩니다.
> 레코드를 생성하기 전에 `READY` 상태가 될 때까지 기다려야 합니다(일반적으로 25~30초).

In [ ]:
create_resp = cp_client.create_registry(
    name="enterprise_agent_registry",
    description="Enterprise registry for MCP servers, A2A agents, and custom resources. Manual approval required.",
    approvalConfiguration={"autoApproval": False},
)

REGISTRY_ARN = create_resp["registryArn"]
REGISTRY_ID = REGISTRY_ARN.split("/")[-1]

print(f"{C.GREEN}✅ Registry created!{C.RESET}")
print(f"  {C.BOLD}Name:{C.RESET}          enterprise_agent_registry")
print(f"  {C.BOLD}ID:{C.RESET}            {C.CYAN}{REGISTRY_ID}{C.RESET}")
print(
    f"  {C.BOLD}Auto-Approval:{C.RESET}  {C.RED}{C.BOLD}False{C.RESET} {C.DIM}(all records require manual approval){C.RESET}"
)
print(f"\n{C.YELLOW}⏳ Waiting for READY status...{C.RESET}")

wait_for_registry_ready(cp_client, REGISTRY_ID)

### 검증: GetRegistry 및 ListRegistries

In [ ]:
registry = cp_client.get_registry(registryId=REGISTRY_ID)

print(f"{C.BOLD}=== Registry Details ==={C.RESET}\n")
print(f"  {C.BOLD}Name:{C.RESET}           {registry['name']}")
print(f"  {C.BOLD}Status:{C.RESET}         {C.GREEN}{registry['status']}{C.RESET}")
auto_val = registry.get("approvalConfiguration", {}).get("autoApproval", "N/A")
auto_color = C.RED if not auto_val else C.GREEN
print(f"  {C.BOLD}Auto-Approval:{C.RESET}  {auto_color}{C.BOLD}{auto_val}{C.RESET}")
print(f"  {C.BOLD}Created:{C.RESET}        {registry.get('createdAt', 'N/A')}")

print(f"\n{C.BOLD}=== All Registries ==={C.RESET}\n")
for reg in cp_client.list_registries()["registries"]:
    status_color = C.GREEN if reg["status"] == "READY" else C.YELLOW
    auto = reg.get("approvalConfiguration", {}).get("autoApproval", "N/A")
    print(f"  {status_color}[{reg['status']}]{C.RESET} {reg['name']} {C.DIM}(autoApproval={auto}){C.RESET}")

---
## 2단계 — 각 페르소나의 IAM 정책

각 페르소나에는 범위가 지정된 IAM 정책이 부여됩니다. 핵심 규칙은 간단합니다.

> **관리자만 레코드를 승인하거나 거부할 수 있습니다.** 게시자는 레코드를 생성하고 제출할 수 있지만 자신이 제출한 항목을 승인할 수 없습니다.

각 페르소나가 호출할 수 있는 작업은 다음과 같습니다.

| API 작업 | 관리자 | 게시자 | 소비자 |
|------------|:-----:|:---------:|:--------:|
| `CreateRegistry` / `DeleteRegistry` | ✅ | — | — |
| `CreateRegistryRecord` | ✅ | ✅ | — |
| `UpdateRegistryRecord`(DRAFT만) | ✅ | ✅ | — |
| `SubmitRegistryRecordForApproval` | ✅ | ✅ | — |
| `UpdateRegistryRecordStatus`(승인/거부) | ✅ | — | — |
| `GetRegistryRecord` / `ListRegistryRecords` | ✅ | ✅ | ✅ |
| `SearchRegistryRecords` | ✅ | — | ✅ |

5단계(거버넌스 테스트)에서 이러한 경계를 검증합니다.

### 2.1 관리자 정책 — 전체 액세스 + 승인 권한

In [ ]:
ADMIN_POLICY = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AllowCreatingAndListingRegistries",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:CreateRegistry",
                "bedrock-agentcore:ListRegistries",
            ],
            "Resource": [f"arn:aws:bedrock-agentcore:{AWS_REGION}:{ACCOUNT_ID}:*"],
        },
        {
            "Sid": "AllowGetUpdateDeleteRegistry",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:GetRegistry",
                "bedrock-agentcore:UpdateRegistry",
                "bedrock-agentcore:DeleteRegistry",
            ],
            "Resource": [f"arn:aws:bedrock-agentcore:{AWS_REGION}:{ACCOUNT_ID}:registry/*"],
        },
        {
            "Sid": "AllowCreatingAndListingRegistryRecords",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:CreateRegistryRecord",
                "bedrock-agentcore:ListRegistryRecords",
            ],
            "Resource": [f"arn:aws:bedrock-agentcore:{AWS_REGION}:{ACCOUNT_ID}:registry/*"],
        },
        {
            "Sid": "AllowRecordLevelOperations",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:GetRegistryRecord",
                "bedrock-agentcore:UpdateRegistryRecord",
                "bedrock-agentcore:DeleteRegistryRecord",
                "bedrock-agentcore:SubmitRegistryRecordForApproval",
            ],
            "Resource": [f"arn:aws:bedrock-agentcore:{AWS_REGION}:{ACCOUNT_ID}:registry/*/record/*"],
        },
        {
            "Sid": "AllowApproveRejectDeprecateRecords",
            "Effect": "Allow",
            "Action": ["bedrock-agentcore:UpdateRegistryRecordStatus"],
            "Resource": [f"arn:aws:bedrock-agentcore:{AWS_REGION}:{ACCOUNT_ID}:registry/*/record/*"],
        },
    ],
}
print(f"{C.GREEN}✅ Admin policy defined{C.RESET} — includes {C.BOLD}ApprovalAuthority{C.RESET} (approve/reject).")

### 2.2 게시자 정책 — 생성 및 제출 가능, 승인 불가

In [ ]:
PUBLISHER_POLICY = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AllowListingAllRegistries",
            "Effect": "Allow",
            "Action": ["bedrock-agentcore:ListRegistries"],
            "Resource": [f"arn:aws:bedrock-agentcore:{AWS_REGION}:{ACCOUNT_ID}:*"],
        },
        {
            "Sid": "AllowGetRegistry",
            "Effect": "Allow",
            "Action": ["bedrock-agentcore:GetRegistry"],
            "Resource": [f"arn:aws:bedrock-agentcore:{AWS_REGION}:{ACCOUNT_ID}:registry/*"],
        },
        {
            "Sid": "AllowCreatingAndListingRegistryRecords",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:CreateRegistryRecord",
                "bedrock-agentcore:ListRegistryRecords",
            ],
            "Resource": [f"arn:aws:bedrock-agentcore:{AWS_REGION}:{ACCOUNT_ID}:registry/*"],
        },
        {
            "Sid": "AllowRecordLevelOperations",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:GetRegistryRecord",
                "bedrock-agentcore:UpdateRegistryRecord",
                "bedrock-agentcore:DeleteRegistryRecord",
                "bedrock-agentcore:SubmitRegistryRecordForApproval",
            ],
            "Resource": [f"arn:aws:bedrock-agentcore:{AWS_REGION}:{ACCOUNT_ID}:registry/*/record/*"],
        },
    ],
}
print(f"{C.GREEN}✅ Publisher policy defined{C.RESET} — {C.RED}NO ApprovalAuthority{C.RESET} (cannot self-approve).")

### 2.3 소비자 정책 — 읽기 전용

In [ ]:
CONSUMER_POLICY = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AllowListingAllRegistries",
            "Effect": "Allow",
            "Action": ["bedrock-agentcore:ListRegistries"],
            "Resource": [f"arn:aws:bedrock-agentcore:{AWS_REGION}:{ACCOUNT_ID}:*"],
        },
        {
            "Sid": "AllowGetRegistry",
            "Effect": "Allow",
            "Action": ["bedrock-agentcore:GetRegistry"],
            "Resource": [f"arn:aws:bedrock-agentcore:{AWS_REGION}:{ACCOUNT_ID}:registry/*"],
        },
        {
            "Sid": "AllowSearchingForApprovedRecords",
            "Effect": "Allow",
            "Action": ["bedrock-agentcore:SearchRegistryRecords"],
            "Resource": [f"arn:aws:bedrock-agentcore:{AWS_REGION}:{ACCOUNT_ID}:registry/*"],
        },
        {
            "Sid": "AllowListingAndGettingRecords",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:ListRegistryRecords",
                "bedrock-agentcore:GetRegistryRecord",
            ],
            "Resource": [f"arn:aws:bedrock-agentcore:{AWS_REGION}:{ACCOUNT_ID}:registry/*"],
        },
    ],
}
print(f"{C.GREEN}✅ Consumer policy defined{C.RESET} — {C.CYAN}read-only{C.RESET}, no write access.")

---
## 3단계 — IAM 사용자 생성 및 정책 연결

세 명의 IAM 사용자(페르소나별 한 명)를 생성하고 각 인라인 정책을 연결합니다.
각 사용자의 관점에서 API 호출을 테스트할 수 있도록 액세스 키도 생성합니다.

> **참고**: IAM 정책 전파에는 약 10초가 걸립니다. 정책을 연결한 후 잠시 기다립니다.

In [ ]:
USERS = {
    "registry-admin-demo": ADMIN_POLICY,
    "registry-publisher-demo": PUBLISHER_POLICY,
    "registry-consumer-demo": CONSUMER_POLICY,
}

PERSONA_LABELS = {
    "registry-admin-demo": "Admin",
    "registry-publisher-demo": "Publisher",
    "registry-consumer-demo": "Consumer",
}

user_credentials = {}

for user_name, policy in USERS.items():
    try:
        iam_client.create_user(UserName=user_name)
    except iam_client.exceptions.EntityAlreadyExistsException:
        pass

    policy_name = f"{user_name}-policy"
    iam_client.put_user_policy(UserName=user_name, PolicyName=policy_name, PolicyDocument=json.dumps(policy))

    try:
        keys = iam_client.create_access_key(UserName=user_name)
    except ClientError as e:
        if "LimitExceeded" in str(e):
            for k in iam_client.list_access_keys(UserName=user_name)["AccessKeyMetadata"]:
                iam_client.delete_access_key(UserName=user_name, AccessKeyId=k["AccessKeyId"])
            keys = iam_client.create_access_key(UserName=user_name)
        else:
            raise

    user_credentials[user_name] = {
        "access_key": keys["AccessKey"]["AccessKeyId"],
        "secret_key": keys["AccessKey"]["SecretAccessKey"],
    }

# 표로 표시(보안 정보는 표시하지 않음)
print(f"{C.BOLD}=== IAM Users Created ==={C.RESET}\n")
print(f"  {C.BOLD}{'Persona':<12} {'IAM User':<28} {'Policy':<35}{C.RESET}")
print(f"  {'─' * 12} {'─' * 28} {'─' * 35}")
for user_name in USERS:
    persona = PERSONA_LABELS[user_name]
    policy_name = f"{user_name}-policy"
    print(f"  {C.CYAN}{persona:<12}{C.RESET} {user_name:<28} {policy_name:<35}")

print(f"\n{C.YELLOW}⏳ Waiting 10s for IAM policy propagation...{C.RESET}")
time.sleep(10)
print(f"{C.GREEN}✅ All 3 users ready.{C.RESET}")

In [ ]:
def get_client_for_user(user_name, service="bedrock-agentcore-control"):
    """
    지정한 IAM 사용자로 인증된 boto3 클라이언트를 생성합니다.

    매개변수:
        user_name: IAM 사용자 이름(예: "registry-publisher-demo")
        service:   boto3 서비스 이름(컨트롤 플레인 또는 데이터 플레인)
    반환값:
        지정한 사용자로 인증된 boto3 클라이언트
    """
    creds = user_credentials[user_name]
    s = boto3.Session(
        aws_access_key_id=creds["access_key"],
        aws_secret_access_key=creds["secret_key"],
        region_name=AWS_REGION,
    )
    return s.client(service)


# 페르소나별 클라이언트 생성(Notebook의 나머지 부분에서 사용)
publisher_cp = get_client_for_user("registry-publisher-demo")
admin_cp = get_client_for_user("registry-admin-demo")
consumer_cp = get_client_for_user("registry-consumer-demo")
consumer_dp = get_client_for_user("registry-consumer-demo", service="bedrock-agentcore")

print(f"{C.GREEN}✅ Per-persona clients ready:{C.RESET}")
print(f"  {C.CYAN}publisher_cp{C.RESET}  → Publisher (control plane)")
print(f"  {C.CYAN}admin_cp{C.RESET}      → Admin (control plane)")
print(f"  {C.CYAN}consumer_cp{C.RESET}   → Consumer (control plane)")
print(f"  {C.CYAN}consumer_dp{C.RESET}   → Consumer (data plane / search)")

---
## 4단계 — 모든 레코드 생성(MCP, A2A, CUSTOM)

게시자가 각 유형의 레코드를 하나씩 생성합니다. 모든 레코드는 `DRAFT` 상태로 시작합니다.
다음 단계의 거버넌스 테스트에 같은 레코드를 사용하므로 중복 레코드는 생성하지 않습니다.

### 4a. MCP Server 레코드

In [ ]:
mcp_rec = publisher_cp.create_registry_record(
    registryId=REGISTRY_ID,
    name="enterprise_code_review_mcp",
    descriptorType="MCP",
    descriptors={
        "mcp": {
            "server": {
                "inlineContent": json.dumps(
                    {
                        "name": "io.enterprise/code-review",
                        "description": "MCP server for automated code review with security scanning",
                        "version": "2.1.0",
                        "packages": [
                            {
                                "registryType": "npm",
                                "identifier": "@enterprise/code-review-mcp",
                                "version": "2.1.0",
                                "transport": {"type": "stdio"},
                            }
                        ],
                    }
                )
            },
            "tools": {
                "inlineContent": json.dumps(
                    {
                        "tools": [
                            {
                                "name": "review_code",
                                "description": "Analyze code for security issues",
                                "inputSchema": {
                                    "type": "object",
                                    "properties": {"code": {"type": "string"}},
                                },
                            }
                        ]
                    }
                )
            },
        }
    },
    recordVersion="2.1",
)
RECORD_IDS["mcp"] = mcp_rec["recordArn"].split("/")[-1]
RECORD_NAMES["mcp"] = "enterprise_code_review_mcp"

print(f"{C.GREEN}✅ MCP record created{C.RESET}")
print(f"  {C.BOLD}Name:{C.RESET}     enterprise_code_review_mcp")
print(f"  {C.BOLD}Type:{C.RESET}     MCP")
print(f"  {C.BOLD}ID:{C.RESET}       {C.CYAN}{RECORD_IDS['mcp']}{C.RESET}")
print(f"  {C.BOLD}Status:{C.RESET}   {C.YELLOW}DRAFT{C.RESET}")

### 4b. A2A Agent 레코드(인라인 Agent Card)

A2A 레코드는 인라인 agent card 설명자를 사용하므로 외부 URL 동기화가 필요하지 않습니다.
agent card JSON은 `descriptors` 필드에 직접 제공됩니다.

In [ ]:
a2a_rec = publisher_cp.create_registry_record(
    registryId=REGISTRY_ID,
    name="enterprise_compliance_agent",
    descriptorType="A2A",
    descriptors={
        "a2a": {
            "agentCard": {
                "schemaVersion": "0.3",
                "inlineContent": json.dumps(
                    {
                        "protocolVersion": "0.3",
                        "name": "enterprise-compliance-agent",
                        "description": "A2A agent for compliance policy validation and audit checks",
                        "version": "1.0.0",
                        "url": "https://compliance-agent.internal.example.com/a2a",
                        "capabilities": {"streaming": True},
                        "skills": [
                            {
                                "id": "validate_compliance",
                                "name": "Compliance Validation",
                                "description": "Validates resources against compliance policies",
                                "tags": ["compliance"],
                            },
                            {
                                "id": "audit_check",
                                "name": "Audit Check",
                                "description": "Runs audit checks on infrastructure and configurations",
                                "tags": ["audit"],
                            },
                        ],
                        "defaultInputModes": ["text/plain"],
                        "defaultOutputModes": ["text/plain"],
                    }
                ),
            }
        }
    },
    recordVersion="1.0",
)
RECORD_IDS["a2a"] = a2a_rec["recordArn"].split("/")[-1]
RECORD_NAMES["a2a"] = "enterprise_compliance_agent"

print(f"{C.GREEN}\u2705 A2A record created{C.RESET}")
print(f"  {C.BOLD}Name:{C.RESET}     enterprise_compliance_agent")
print(f"  {C.BOLD}Type:{C.RESET}     A2A")
print(f"  {C.BOLD}ID:{C.RESET}       {C.CYAN}{RECORD_IDS['a2a']}{C.RESET}")
print(f"  {C.BOLD}Status:{C.RESET}   {C.YELLOW}DRAFT{C.RESET}")

### 4c. CUSTOM 리소스 레코드

In [ ]:
custom_rec = publisher_cp.create_registry_record(
    registryId=REGISTRY_ID,
    name="enterprise_data_pipeline_skill",
    descriptorType="CUSTOM",
    descriptors={
        "custom": {
            "inlineContent": json.dumps(
                {
                    "name": "data-pipeline-orchestrator",
                    "description": "Custom skill for orchestrating cross-account data pipelines",
                    "version": "3.0.0",
                    "endpoint": "https://pipelines.internal.example.com/api/v3",
                    "auth": {
                        "type": "IAM",
                        "roleArn": "arn:aws:iam::123456789012:role/pipeline-invoker",
                    },
                }
            )
        }
    },
    recordVersion="3.0",
)
RECORD_IDS["custom"] = custom_rec["recordArn"].split("/")[-1]
RECORD_NAMES["custom"] = "enterprise_data_pipeline_skill"

print(f"{C.GREEN}✅ CUSTOM record created{C.RESET}")
print(f"  {C.BOLD}Name:{C.RESET}     enterprise_data_pipeline_skill")
print(f"  {C.BOLD}Type:{C.RESET}     CUSTOM")
print(f"  {C.BOLD}ID:{C.RESET}       {C.CYAN}{RECORD_IDS['custom']}{C.RESET}")
print(f"  {C.BOLD}Status:{C.RESET}   {C.YELLOW}DRAFT{C.RESET}")

# 요약 표
print(f"\n{C.BOLD}=== All Records Created ==={C.RESET}\n")
print(f"  {C.BOLD}{'Type':<10} {'Name':<35} {'Status':<20}{C.RESET}")
print(f"  {'─' * 10} {'─' * 35} {'─' * 20}")
for rtype, rid in RECORD_IDS.items():
    name = RECORD_NAMES[rtype]
    print(f"  {C.CYAN}{rtype.upper():<10}{C.RESET} {name:<35} {C.YELLOW}DRAFT{C.RESET}")

print(f"\n{C.YELLOW}⏳ Waiting 5s for records to settle...{C.RESET}")
time.sleep(5)
print(f"{C.GREEN}✅ Ready for governance tests.{C.RESET}")

---
## 5단계 — 거버넌스 가드레일 테스트

이 섹션은 Notebook에서 가장 중요합니다. 각 페르소나가 수행할 **수 있는** 작업과 수행해서는 **안 되는** 작업을 테스트하여 IAM 경계가 올바르게 작동하는지 입증합니다.

위에서 생성한 MCP 레코드를 승인 워크플로의 테스트 대상으로 사용합니다.

### 검증할 항목

| 테스트 | 페르소나 | 예상 결과 |
|------|---------|----------|
| 5a. 게시자가 레코드 승인 요청 | 게시자 | ✅ 허용 |
| 5b. 게시자가 자체 승인 시도 | 게시자 | 🚫 거부 |
| 5c. 소비자가 레코드 생성 시도 | 소비자 | 🚫 거부 |
| 5d. 소비자가 레코드 승인 시도 | 소비자 | 🚫 거부 |
| 5e. 소비자가 레코드 조회 | 소비자 | ✅ 허용 |
| 5f. 관리자가 레코드 승인 | 관리자 | ✅ 허용 |

### 5a. 게시자가 MCP 레코드의 승인 요청

`DRAFT → PENDING_APPROVAL` — 게시자가 담당하는 작업입니다.

In [ ]:
mcp_record_id = RECORD_IDS["mcp"]
mcp_name = RECORD_NAMES["mcp"]

print(f"{C.BOLD}=== 5a. Publisher submits MCP record for approval ==={C.RESET}")
print(f"  Record: {C.CYAN}{mcp_name}{C.RESET}\n")
test_action(
    "SubmitRegistryRecordForApproval (DRAFT → PENDING_APPROVAL)",
    lambda: publisher_cp.submit_registry_record_for_approval(registryId=REGISTRY_ID, recordId=mcp_record_id),
)

rec = admin_cp.get_registry_record(registryId=REGISTRY_ID, recordId=mcp_record_id)
print(f"\n  {C.BOLD}Record:{C.RESET} {mcp_name}  →  {C.YELLOW}{rec['status']}{C.RESET}")

### 5b. 게시자가 자체 승인 시도(실패해야 함)

이 테스트는 핵심 거버넌스 테스트입니다. 게시자의 IAM 정책에는
`UpdateRegistryRecordStatus`가 포함되지 않으므로 이 호출은 거부되어야 합니다.

In [ ]:
print(f"{C.BOLD}=== 5b. Publisher tries to self-approve (SHOULD FAIL) ==={C.RESET}")
print(f"  Record: {C.CYAN}{mcp_name}{C.RESET}\n")
test_action(
    "UpdateRegistryRecordStatus → APPROVED (self-approval attempt)",
    lambda: publisher_cp.update_registry_record_status(
        registryId=REGISTRY_ID,
        recordId=mcp_record_id,
        status="APPROVED",
        statusReason="Self-approval attempt",
    ),
)

rec = admin_cp.get_registry_record(registryId=REGISTRY_ID, recordId=mcp_record_id)
print(f"\n  {C.BOLD}Record:{C.RESET} {mcp_name}  →  Status still: {C.YELLOW}{rec['status']}{C.RESET}")
assert rec["status"] == "PENDING_APPROVAL", "GOVERNANCE FAILURE: Publisher was able to self-approve!"
print(f"  {C.GREEN}✅ Governance guardrail PASSED — Publisher cannot self-approve.{C.RESET}")

### 5c. 소비자가 레코드 생성 시도(실패해야 함)

소비자는 읽기 전용입니다. 항목을 생성, 수정 또는 승인할 수 없습니다.

In [ ]:
print(f"{C.BOLD}=== 5c. Consumer tries to create a record (SHOULD FAIL) ==={C.RESET}\n")
test_action(
    "CreateRegistryRecord",
    lambda: consumer_cp.create_registry_record(
        registryId=REGISTRY_ID,
        name="shouldFail",
        descriptorType="MCP",
        descriptors={"mcp": {"server": {"inlineContent": "{}"}}},
        recordVersion="1.0",
    ),
)

print(f"\n{C.BOLD}=== 5d. Consumer tries to approve a record (SHOULD FAIL) ==={C.RESET}")
print(f"  Record: {C.CYAN}{mcp_name}{C.RESET}\n")
test_action(
    "UpdateRegistryRecordStatus → APPROVED",
    lambda: consumer_cp.update_registry_record_status(
        registryId=REGISTRY_ID,
        recordId=mcp_record_id,
        status="APPROVED",
        statusReason="Consumer approval attempt",
    ),
)

### 5e. 소비자가 레코드 조회(성공해야 함)

소비자의 읽기 전용 액세스가 예상대로 작동하는지 확인합니다.

In [ ]:
print(f"{C.BOLD}=== 5e. Consumer read operations (SHOULD SUCCEED) ==={C.RESET}\n")

test_action("ListRegistries", lambda: consumer_cp.list_registries())
test_action("GetRegistry", lambda: consumer_cp.get_registry(registryId=REGISTRY_ID))
test_action(
    "ListRegistryRecords",
    lambda: consumer_cp.list_registry_records(registryId=REGISTRY_ID),
)

rec = test_action(
    f"GetRegistryRecord ({mcp_name})",
    lambda: consumer_cp.get_registry_record(registryId=REGISTRY_ID, recordId=mcp_record_id),
)

if rec:
    print(f"\n  {C.BOLD}Record Details (Consumer view):{C.RESET}")
    print(f"    Name:     {rec['name']}")
    print(f"    Type:     {rec['descriptorType']}")
    print(f"    Status:   {C.YELLOW}{rec['status']}{C.RESET}")

### 5f. 관리자가 MCP 레코드 승인(성공해야 함)

관리자만 `UpdateRegistryRecordStatus` 권한을 갖습니다. 이 작업으로 승인 워크플로가 완료됩니다.

In [ ]:
print(f"{C.BOLD}=== 5f. Admin approves the MCP record ==={C.RESET}")
print(f"  Record: {C.CYAN}{mcp_name}{C.RESET}\n")
test_action(
    "Admin: UpdateRegistryRecordStatus → APPROVED",
    lambda: admin_cp.update_registry_record_status(
        registryId=REGISTRY_ID,
        recordId=mcp_record_id,
        status="APPROVED",
        statusReason="Approved by admin after review",
    ),
)

rec = admin_cp.get_registry_record(registryId=REGISTRY_ID, recordId=mcp_record_id)
print(f"\n  {C.BOLD}Record:{C.RESET} {mcp_name}  →  {C.GREEN}{rec['status']}{C.RESET}")

---
## 6단계 — 나머지 레코드 승인 및 시맨틱 검색

위의 거버넌스 테스트에서 MCP 레코드를 승인했습니다. 이제 A2A와 CUSTOM 레코드의 승인을 요청하고 승인한 다음,
세 레코드를 모두 시맨틱 검색으로 찾을 수 있는지 확인합니다.

In [ ]:
# A2A 및 CUSTOM 레코드의 승인 요청
print(f"{C.BOLD}=== Submit remaining records for approval ==={C.RESET}\n")
for rtype in ["a2a", "custom"]:
    rid = RECORD_IDS[rtype]
    name = RECORD_NAMES[rtype]
    test_action(
        f"Submit {name} ({rtype.upper()})",
        lambda rid=rid: publisher_cp.submit_registry_record_for_approval(registryId=REGISTRY_ID, recordId=rid),
    )

print(f"\n{C.YELLOW}⏳ Waiting 5s for records to settle...{C.RESET}")
time.sleep(5)

# 관리자가 두 레코드 모두 승인
print(f"\n{C.BOLD}=== Admin approves remaining records ==={C.RESET}\n")
for rtype in ["a2a", "custom"]:
    rid = RECORD_IDS[rtype]
    name = RECORD_NAMES[rtype]
    test_action(
        f"Approve {name} ({rtype.upper()})",
        lambda rid=rid: admin_cp.update_registry_record_status(
            registryId=REGISTRY_ID,
            recordId=rid,
            status="APPROVED",
            statusReason="Approved by admin",
        ),
    )

# 최종 상태 표
print(f"\n{C.BOLD}=== Final Record Status ==={C.RESET}\n")
print(f"  {C.BOLD}{'Type':<10} {'Name':<35} {'Status':<15}{C.RESET}")
print(f"  {'─' * 10} {'─' * 35} {'─' * 15}")
for rtype, rid in RECORD_IDS.items():
    r = admin_cp.get_registry_record(registryId=REGISTRY_ID, recordId=rid)
    name = RECORD_NAMES[rtype]
    status = r["status"]
    sc = C.GREEN if status == "APPROVED" else C.YELLOW
    print(f"  {C.CYAN}{rtype.upper():<10}{C.RESET} {name:<35} {sc}{status}{C.RESET}")

print(f"\n{C.GREEN}✅ All records approved.{C.RESET}")

### 시맨틱 검색 검증(Data Plane)

레코드가 APPROVED 상태가 되면 data plane의 `SearchRegistryRecords` API로 검색할 수 있습니다.

> **참고**: 검색은 최종 일관성을 따릅니다. 승인 후 결과가 표시되는 데 몇 초가 걸릴 수 있습니다.

In [ ]:
print(f"{C.YELLOW}⏳ Waiting 30s for search index propagation...{C.RESET}")
time.sleep(30)

queries = [
    "code review security scanning",
    "compliance policy validation",
    "data pipeline orchestration",
]

# 검색에는 data plane 클라이언트(bedrock-agentcore) 사용

for q in queries:
    print(f"\n{C.BOLD}🔍 Search: '{q}'{C.RESET}")
    try:
        results = consumer_dp.search_registry_records(registryIds=[REGISTRY_ARN], searchQuery=q, maxResults=5)
        if results.get("registryRecords"):
            for r in results["registryRecords"]:
                print(f"  {C.GREEN}✅{C.RESET} [{C.CYAN}{r.get('descriptorType', 'N/A')}{C.RESET}] {r['name']}")
        else:
            print(f"  {C.YELLOW}⏳ No results yet (index may still be propagating){C.RESET}")
    except Exception as e:
        print(f"  {C.RED}Error: {e}{C.RESET}")

---
## 7단계 — 프로덕션 준비 체크리스트

프로덕션으로 전환하기 전에 다음 항목을 확인하세요.

### 레지스트리 구성
- [ ] 모든 프로덕션 레지스트리에 `autoApproval: false`가 설정되어 있음
- [ ] 레지스트리 이름이 조직의 명명 규칙을 따름
- [ ] 레지스트리 설명에 담당 팀과 목적이 포함되어 있음

### IAM 정책
- [ ] 관리자 정책에 승인 권한인 `UpdateRegistryRecordStatus`가 포함되어 있음
- [ ] 게시자 정책에 `UpdateRegistryRecordStatus`가 포함되어 있지 **않음**
- [ ] 승인된 레코드의 편집을 방지하도록 게시자 정책의 `UpdateRegistryRecord`에 조건 키가 있음
- [ ] 소비자 정책이 엄격한 읽기 전용임(`Create*`, `Update*`, `Delete*`, `Submit*` 없음)
- [ ] 모든 정책에서 리소스 ARN으로 범위를 지정함(`*` 사용 안 함)
- [ ] 조직 표준에 따라 인라인 정책 또는 관리형 정책을 사용함

### 거버넌스
- [ ] 자체 승인 방지 테스트 완료: 게시자가 `UpdateRegistryRecordStatus`를 호출할 수 없음
- [ ] 승인 워크플로 엔드 투 엔드 테스트 완료: DRAFT → PENDING_APPROVAL → APPROVED
- [ ] 거부 워크플로 테스트 완료: PENDING_APPROVAL → REJECTED
- [ ] 사용 중단 워크플로 테스트 완료: APPROVED → DEPRECATED

### 운영
- [ ] Data plane을 통해 승인된 레코드가 검색됨
- [ ] 감사 추적을 위한 CloudTrail 로깅이 활성화되어 있음
- [ ] 레코드 승인, 거부, 사용 중단, 긴급 레코드 제거에 관한 Runbook이 문서화되어 있음
- [ ] 긴급 제거 시 `DeleteRegistryRecord`를 사용하는 방법을 온콜 팀이 알고 있음

### 레코드 유형
- [ ] `server` + `tools` 설명자를 사용하는 MCP 서버 레코드 테스트 완료
- [ ] `agentCard`를 사용하는 A2A 에이전트 레코드 테스트 완료
- [ ] `inlineContent`를 사용하는 CUSTOM 레코드 테스트 완료(해당하는 경우)

---
## 8단계 — 문제 해결 FAQ

### Q: 레코드 생성 시 `ConflictException` 발생
**A**: 레지스트리가 아직 `CREATING` 또는 `UPDATING` 상태입니다. 레코드를 생성하기 전에 `READY` 상태가 될 때까지 기다리세요. `wait_for_registry_ready()` 헬퍼를 사용합니다.

### Q: 승인 요청 시 `ConflictException` 발생
**A**: `CreateRegistryRecord` 호출 후 약 5초 동안 기다린 다음 `SubmitRegistryRecordForApproval`을 호출하세요. 레코드가 안정된 상태가 되는 데 시간이 필요합니다.

### Q: 레코드 승인 시 `ValidationException` 발생
**A**: 레코드가 `PENDING_APPROVAL` 상태가 아닙니다. 먼저 `GetRegistryRecord`로 확인하세요. 제출된 레코드만 승인할 수 있습니다.

### Q: 게시자가 레코드를 승인할 수 있음(거버넌스 우회)
**A**: 게시자의 IAM 정책에 `UpdateRegistryRecordStatus`가 잘못 포함되어 있습니다. 이 작업을 제거하세요. 관리자 정책에만 이 작업이 있어야 합니다.

### Q: 레코드를 승인한 후 검색 결과가 0건임
**A**: 검색은 최종 일관성을 따릅니다. 승인 후 10~30초 동안 기다리세요. 테스트 중에는 전파에 더 오랜 시간이 걸릴 수 있습니다. 결과가 0건이라고 해서 즉시 실패로 간주하지 마세요.

### Q: 액세스 권한이 있어야 하는 사용자에게 `AccessDeniedException` 발생
**A**: `PutUserPolicy` 호출 후 IAM 정책 전파에는 약 10초가 걸립니다. 잠시 기다린 후 다시 시도하세요. 리소스 ARN 패턴이 레지스트리와 일치하는지도 확인하세요.

### Q: 이미 승인된 레코드를 업데이트하려면 어떻게 해야 하나요?
**A**: `UpdateRegistryRecord`를 호출하면 레코드가 `DRAFT` 상태로 재설정됩니다. 승인 워크플로를 다시 진행해야 합니다. 정책이 승인 상태에 `StringNotEqualsIfExists`를 사용한다면 게시자 조건 키로 이 작업이 차단됩니다.

---
## 9단계 — 정리

이 가이드에서 생성한 모든 리소스를 제거합니다. 다음 순서를 지켜야 합니다.
1. 레지스트리 레코드(모두)
2. 레지스트리
3. IAM 액세스 키 → 인라인 정책 → 사용자

In [ ]:
print(f"{C.BOLD}=== Cleanup: Registry Records ==={C.RESET}\n")
for rtype, rid in RECORD_IDS.items():
    name = RECORD_NAMES.get(rtype, rtype)
    try:
        cp_client.delete_registry_record(registryId=REGISTRY_ID, recordId=rid)
        print(f"  {C.GREEN}✅{C.RESET} Deleted record: {C.CYAN}{name}{C.RESET} ({rtype.upper()})")
    except Exception as e:
        print(f"  {C.YELLOW}⚠️{C.RESET}  Record cleanup ({name}): {C.DIM}{e}{C.RESET}")

print(f"\n{C.BOLD}=== Cleanup: Registry ==={C.RESET}\n")
try:
    cp_client.delete_registry(registryId=REGISTRY_ID)
    print(f"  {C.GREEN}✅{C.RESET} Deleted registry: {C.CYAN}enterprise_agent_registry{C.RESET}")
except Exception as e:
    print(f"  {C.YELLOW}⚠️{C.RESET}  Registry cleanup: {C.DIM}{e}{C.RESET}")

print(f"\n{C.BOLD}=== Cleanup: IAM Users ==={C.RESET}\n")
for user_name in USERS.keys():
    persona = PERSONA_LABELS.get(user_name, user_name)
    try:
        for k in iam_client.list_access_keys(UserName=user_name)["AccessKeyMetadata"]:
            iam_client.delete_access_key(UserName=user_name, AccessKeyId=k["AccessKeyId"])
        iam_client.delete_user_policy(UserName=user_name, PolicyName=f"{user_name}-policy")
        iam_client.delete_user(UserName=user_name)
        print(f"  {C.GREEN}✅{C.RESET} Deleted user: {C.CYAN}{persona}{C.RESET} ({user_name})")
    except Exception as e:
        print(f"  {C.YELLOW}⚠️{C.RESET}  User cleanup ({persona}): {C.DIM}{e}{C.RESET}")

print(f"\n{C.GREEN}✅ Cleanup complete.{C.RESET}")